In [36]:

import pandas as pd

df_2017 = pd.read_csv("Data.raw/CHC_2017/Estructura CHC_2017.csv", sep=";", encoding="latin-1")
print(df_2017.shape)
df_2017.head()

(9538, 123)


,ï»¿DIRECTORIO,TIP_FOR,P1,P1S1,P2,P2S1,P5,CTL_1,P8,P9R,...,P35,P36R,P37S1,P37S2,P37S3,P37S4,P37S5,P37S6,P37S7,COMPLETA
0,101000,1,11,11001,1,16,2,1,38,1,...,,,,,,,,,,1
1,101001,1,11,11001,1,16,2,1,38,2,...,,,,,,,,,,1
2,101002,1,11,11001,1,16,2,1,25,2,...,,,,,,,,,,1
3,101003,1,11,11001,1,16,2,1,52,1,...,,,,,,,,,,1
4,101004,1,11,11001,1,16,2,1,27,2,...,,,,,,,,,,1


In [37]:
df_2021 = pd.read_csv("Data.raw/CHC_2021 (1)/CHC_base_anonimizada09-09-2021.csv", sep=";", encoding="latin-1")


In [38]:
# Seleccionamos solo las columnas que nos importan para el proyecto
columnas_utiles = {
    "P2S1": "localidad",
    "P8": "edad",
    "P9R": "sexo",
    "P22": "causa_inicio",
    "P24": "causa_persistencia",
    "P23S1R": "anios_en_calle",
    "P27": "sabe_leer_escribir",
    "P28R": "nivel_educativo",
    "P30S1": "consume_cigarrillo",
    "P30S2": "consume_alcohol",
    "P30S3": "consume_marihuana",
    "P30S4": "consume_inhalantes",
    "P30S5": "consume_cocaina",
    "P30S6": "consume_basuco",
    "P30S7": "consume_heroina",
}

df_2017_limpio = df_2017[list(columnas_utiles.keys())].rename(columns=columnas_utiles)
df_2017_limpio.head()

,localidad,edad,sexo,causa_inicio,causa_persistencia,anios_en_calle,sabe_leer_escribir,nivel_educativo,consume_cigarrillo,consume_alcohol,consume_marihuana,consume_inhalantes,consume_cocaina,consume_basuco,consume_heroina
0,16,38,1,1,10,14,1,2,2,2,2,2,2,2,2
1,16,38,2,4,10,4,1,2,2,2,1,2,2,2,2
2,16,25,2,7,7,4,2,13,2,2,2,2,2,2,2
3,16,52,1,7,11,2,2,1,2,2,2,2,2,2,2
4,16,27,2,1,10,0,2,13,1,2,2,1,2,2,2


In [39]:
# Revisamos tipos de datos y nulos
print(df_2017_limpio.dtypes)
print("\n--- Nulos por columna ---")
print(df_2017_limpio.isnull().sum())

localidad             int64
edad                    str
sexo                    str
causa_inicio            str
causa_persistencia      str
anios_en_calle          str
sabe_leer_escribir      str
nivel_educativo         str
consume_cigarrillo      str
consume_alcohol         str
consume_marihuana       str
consume_inhalantes      str
consume_cocaina         str
consume_basuco          str
consume_heroina         str
dtype: object

--- Nulos por columna ---
localidad             0
edad                  0
sexo                  0
causa_inicio          0
causa_persistencia    0
anios_en_calle        0
sabe_leer_escribir    0
nivel_educativo       0
consume_cigarrillo    0
consume_alcohol       0
consume_marihuana     0
consume_inhalantes    0
consume_cocaina       0
consume_basuco        0
consume_heroina       0
dtype: int64


In [40]:
# Vemos los valores únicos de causa_inicio para detectar los "falsos no-nulos"
df_2017_limpio['causa_inicio'].value_counts(dropna=False)

causa_inicio
1     2661
      2622
7     2272
2      518
5      441
4      271
6      253
11     224
10     101
3       89
9       51
8       35
Name: count, dtype: int64

In [41]:
import numpy as np

# Aplicamos strip a todas las columnas de texto, sin importar el nombre exacto del dtype
for col in df_2017_limpio.columns:
    if col != "localidad":  # esta es la única numérica
        df_2017_limpio[col] = df_2017_limpio[col].astype(str).str.strip()
        df_2017_limpio[col] = df_2017_limpio[col].replace("nan", np.nan)

# Reemplazamos strings vacíos "" por nulos reales
df_2017_limpio = df_2017_limpio.replace("", np.nan)

# Verificamos de nuevo
print(df_2017_limpio.isnull().sum())

localidad                0
edad                  2592
sexo                  2592
causa_inicio          2622
causa_persistencia    2630
anios_en_calle        2629
sabe_leer_escribir    2641
nivel_educativo       2645
consume_cigarrillo    2646
consume_alcohol       2647
consume_marihuana     2646
consume_inhalantes    2648
consume_cocaina       2648
consume_basuco        2648
consume_heroina       2649
dtype: int64


In [42]:
mapa_causa_persistencia = {
    "1": "Consumo de sustancias psicoactivas",
    "2": "Por gusto personal",
    "3": "Las amistades",
    "4": "Dificultades económicas",
    "5": "Falta de trabajo",
    "6": "Enfermedad",
    "7": "Conflictos o dificultades familiares",
    "8": "Siempre ha vivido en la calle",
    "9": "Soledad",
    "10": "Está haciendo proceso en un centro de atención",
    "11": "Otra",
}

df_2017_limpio["causa_persistencia_texto"] = df_2017_limpio["causa_persistencia"].map(mapa_causa_persistencia)
df_2017_limpio["causa_persistencia_texto"].value_counts()


causa_persistencia_texto
Consumo de sustancias psicoactivas                2656
Por gusto personal                                 958
Dificultades económicas                            763
Conflictos o dificultades familiares               709
Falta de trabajo                                   644
Está haciendo proceso en un centro de atención     510
Soledad                                            277
Otra                                               249
Siempre ha vivido en la calle                       57
Las amistades                                       51
Enfermedad                                          34
Name: count, dtype: int64

In [43]:
mapa_si_no = {"1": "Sí", "2": "No"}

columnas_consumo = [
    "consume_cigarrillo", "consume_alcohol", "consume_marihuana",
    "consume_inhalantes", "consume_cocaina", "consume_basuco", "consume_heroina"
]

for col in columnas_consumo:
    df_2017_limpio[col] = df_2017_limpio[col].map(mapa_si_no)

# Vista rápida de consumo por sustancia (% de "Sí" sobre los que respondieron)
for col in columnas_consumo:
    print(col, "->", df_2017_limpio[col].value_counts(normalize=True).round(2).to_dict())

consume_cigarrillo -> {'Sí': 0.75, 'No': 0.25}
consume_alcohol -> {'No': 0.58, 'Sí': 0.42}
consume_marihuana -> {'Sí': 0.56, 'No': 0.44}
consume_inhalantes -> {'No': 0.81, 'Sí': 0.19}
consume_cocaina -> {'No': 0.87, 'Sí': 0.13}
consume_basuco -> {'Sí': 0.66, 'No': 0.34}
consume_heroina -> {'No': 0.97, 'Sí': 0.03}


In [44]:
df_2017_limpio['sexo'].value_counts(dropna=False)


sexo
1      6213
NaN    2592
2       733
Name: count, dtype: int64

In [45]:
df_2017_limpio['edad'] = pd.to_numeric(df_2017_limpio['edad'], errors='coerce')
df_2017_limpio['edad'].describe()

count    6946.000000
mean       39.055716
std        13.448884
min        14.000000
25%        29.000000
50%        37.000000
75%        49.000000
max        90.000000
Name: edad, dtype: float64

In [46]:
mapa_localidad = {
    1: "Usaquén", 2: "Chapinero", 3: "Santa Fe", 4: "San Cristóbal",
    5: "Usme", 6: "Tunjuelito", 7: "Bosa", 8: "Kennedy",
    9: "Fontibón", 10: "Engativá", 11: "Suba", 12: "Barrios Unidos",
    13: "Teusaquillo", 14: "Los Mártires", 15: "Antonio Nariño",
    16: "Puente Aranda", 17: "La Candelaria", 18: "Rafael Uribe Uribe",
    19: "Ciudad Bolívar", 20: "Sumapaz"
}

df_2017_limpio['localidad_texto'] = df_2017_limpio['localidad'].map(mapa_localidad)
df_2017_limpio['localidad_texto'].value_counts()

localidad_texto
Los Mártires          2258
Puente Aranda         1636
Santa Fe              1495
Kennedy                793
Antonio Nariño         498
Engativá               394
Teusaquillo            362
Rafael Uribe Uribe     298
Ciudad Bolívar         292
Chapinero              229
Suba                   217
San Cristóbal          204
Barrios Unidos         194
Usaquén                144
La Candelaria          134
Fontibón               118
Bosa                   111
Tunjuelito              93
Usme                    68
Name: count, dtype: int64

In [47]:
import os
os.makedirs("data_clean", exist_ok=True)


In [48]:
df_2017_limpio.to_csv("data_clean/CHC_2017_limpio.csv", index=False, encoding="utf-8-sig")
print("Guardado exitosamente")

Guardado exitosamente


In [49]:
mapa_causa_inicio = {
    "1": "Consumo de sustancias psicoactivas",
    "2": "Por gusto personal",
    "3": "Amenaza o riesgo para su vida o integridad física",
    "4": "Influencia de otras personas",
    "5": "Dificultades económicas",
    "6": "Falta de trabajo",
    "7": "Conflictos o dificultades familiares",
    "8": "Abuso sexual",
    "9": "Siempre ha vivido en la calle",
    "10": "Víctima del conflicto armado o desplazado",
    "11": "Otra",
}

df_2017_limpio["causa_inicio_texto"] = df_2017_limpio["causa_inicio"].map(mapa_causa_inicio)

# Verificamos que ya esté
print(df_2017_limpio["causa_inicio_texto"].value_counts())

# Volvemos a exportar el CSV completo con la columna ya incluida
df_2017_limpio.to_csv("data_clean/CHC_2017_limpio.csv", index=False, encoding="utf-8-sig")
print("Guardado exitosamente")

causa_inicio_texto
Consumo de sustancias psicoactivas                   2661
Conflictos o dificultades familiares                 2272
Por gusto personal                                    518
Dificultades económicas                               441
Influencia de otras personas                          271
Falta de trabajo                                      253
Otra                                                  224
Víctima del conflicto armado o desplazado             101
Amenaza o riesgo para su vida o integridad física      89
Siempre ha vivido en la calle                          51
Abuso sexual                                           35
Name: count, dtype: int64
Guardado exitosamente


In [51]:
# Transformamos de "ancho" a "largo": una fila por persona-sustancia
columnas_consumo = ["consume_cigarrillo", "consume_alcohol", "consume_marihuana",
                     "consume_inhalantes", "consume_cocaina", "consume_basuco", "consume_heroina"]

df_consumo_2017 = df_2017_limpio.melt(
    value_vars=columnas_consumo,
    var_name="sustancia",
    value_name="consume"
)

# Limpiamos el nombre de la sustancia (quitamos el prefijo "consume_")
df_consumo_2017["sustancia"] = df_consumo_2017["sustancia"].str.replace("consume_", "").str.capitalize()

# Nos quedamos solo con los "Sí" (así el conteo directo en Power BI ya es el total de consumidores)
df_consumo_2017_si = df_consumo_2017[df_consumo_2017["consume"] == "Sí"]

df_consumo_2017_si["sustancia"].value_counts()

sustancia
Cigarrillo    5175
Basuco        4534
Marihuana     3888
Alcohol       2910
Inhalantes    1293
Cocaina        878
Heroina        176
Name: count, dtype: int64

In [52]:
import os

df_consumo_2017_si.to_csv("data_clean/consumo_2017.csv", index=False, encoding="utf-8-sig")


In [53]:
df_2017_limpio.columns.tolist()

['localidad',
 'edad',
 'sexo',
 'causa_inicio',
 'causa_persistencia',
 'anios_en_calle',
 'sabe_leer_escribir',
 'nivel_educativo',
 'consume_cigarrillo',
 'consume_alcohol',
 'consume_marihuana',
 'consume_inhalantes',
 'consume_cocaina',
 'consume_basuco',
 'consume_heroina',
 'causa_persistencia_texto',
 'localidad_texto',
 'causa_inicio_texto']